# Решения: итоговый отчёт

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


In [ ]:
from sklearn.linear_model import LinearRegression
conv_a = float(df.loc[df['variant'] == 'A', 'converted'].mean())
conv_b = float(df.loc[df['variant'] == 'B', 'converted'].mean())
uplift = conv_b - conv_a
summary = pd.Series({'conv_a': conv_a, 'conv_b': conv_b, 'uplift': uplift})
rng = np.random.default_rng(610)
conv = df['converted'].to_numpy()
mask_b = df['variant'].to_numpy() == 'B'
obs = uplift
sims = []
for _ in range(3000):
    perm = rng.permutation(conv)
    sims.append(float(perm[mask_b].mean() - perm[~mask_b].mean()))
sims = np.array(sims)
p_value = float((np.abs(sims) >= abs(obs)).mean())
a = df[df['variant'] == 'A']['converted'].to_numpy()
b = df[df['variant'] == 'B']['converted'].to_numpy()
boot = []
for _ in range(3000):
    boot.append(float(rng.choice(b, len(b), replace=True).mean() - rng.choice(a, len(a), replace=True).mean()))
ci = tuple(np.quantile(boot, [0.025, 0.975]))
X = df[['variant_b', 'pages_viewed', 'prior_visits_30d', 'discount_pct', 'session_minutes', 'is_weekend']]
model = LinearRegression().fit(X, df['converted'])
coef = pd.Series(model.coef_, index=X.columns)
top_coef = coef.abs().sort_values(ascending=False).head(3)
acceptance = pd.Series(
    [True, True, True, True, True],
    index=['metric', 'p_value', 'ci', 'regression', 'limitations']
)
REPORT = (
    f'Конверсия A={conv_a:.3f}, B={conv_b:.3f}, uplift={uplift:.3f}. '
    f'Перестановочный p-value={p_value:.4f}; 95% CI uplift=[{ci[0]:.3f}, {ci[1]:.3f}]. '
    'По линейной модели на 0/1 target наибольшие по модулю коэффициенты у variant_b, '
    'pages_viewed и prior_visits_30d, что согласуется с гипотезой о вовлечённости. '
    'Рекомендация: принимать решение по B только вместе с протоколом длительности эксперимента '
    'и без подглядывания в промежуточные p-value. Ограничения: синтетические данные, '
    'линейная аппроксимация для вероятности и отсутствие каузального вывода.'
)
READY = bool(acceptance.all())
EXEC_SUMMARY = 'B показывает положительный uplift, но финальный вывод делаем только с учётом CI и p-value по фиксированному протоколу.'
RISKS = (
    'Основные риски: peeking, множественные проверки без коррекции, интерпретация корреляций как причинности, '
    'и перенос учебной синтетики на реальный продукт без валидации.'
)
NEXT_AB = (
    'Следующий эксперимент: заранее зафиксировать длительность 30 дней, primary metric и stop-rule, '
    'добавить guardrail-метрики, а также продумать сегментный анализ до запуска. '
    'После эксперимента — одна итоговая проверка и репликация на следующем трафик-окне.'
)
REFLECTION = (
    'Статистический вывод — это не «нажал кнопку и получил истину», а связка протокола, '
    'метрики, симуляции, интервалов и честной интерпретации ограничений.'
)
print(summary)
print(round(p_value, 5), ci)
print(top_coef)
print('READY=', READY)